# Step 14.4 — Feed-Forward MLP inside the Block：Attention 之后为什么还需要 `C → 4C → activation → C`？

Step 14.1–14.3 已经建立：

```text
Residual connection
Normalization
Pre-Norm / Post-Norm placement
```

但 Transformer block 里不只有 Attention。

另一类核心 sublayer 是 **Feed-Forward Network / Feed-Forward MLP（前馈网络 / 多层感知机）**。

经典 Transformer-style 形状可以先记成：

$$
\boxed{
C
\rightarrow
4C
\rightarrow
\text{activation}
\rightarrow
C
}
$$

应用到 sequence tensor：

$$
(B,T,C)
\rightarrow
(B,T,4C)
\rightarrow
(B,T,4C)
\rightarrow
(B,T,C).
$$

本节只理解并实现这个 position-wise MLP；不进入 Step 14.5 的完整 Transformer Block 组装、参数总计与运行测试。


## 1. Attention 和 MLP 到底分工有什么不同？

### Attention

Attention 的核心工作是：

> **cross-position mixing（跨 token / position 混合信息）**。

一个 token 的输出可以从其他合法 sequence positions 的 Values 中取回信息。

### MLP

Feed-Forward MLP 的核心工作是：

> **within-position feature transformation（单个 token 内部的 feature 变换）**。

对：

$$
x\in\mathbb R^{B\times T\times C},
$$

MLP 对每个：

$$
x_{b,t,:}\in\mathbb R^C
$$

独立应用同一套 learned function。

所以 MLP 本身不会计算 token 0 和 token 1 之间的 `T×T` interaction matrix。

但它处理的 token representation 已经可能包含 Attention 从其他 positions 汇入的信息，因此“MLP 不 cross-token mixing”并不等于“MLP 完全不知道上下文”。


## 2. Position-Wise 的精确定义：同一套 MLP 重复应用到所有 Tokens

定义一个 token-level function：

$$
f:\mathbb R^C\rightarrow\mathbb R^C.
$$

Transformer MLP 对 sequence tensor 的规则是：

$$
\boxed{
Y[b,t,:]
=
f\big(X[b,t,:]\big)
}.
$$

同一组参数：

$$
W_1,b_1,W_2,b_2
$$

会共享给所有：

- batch samples；
- sequence positions。

因此参数数量不随 T 增长。

但计算量会随 token 数量增加；parameter sharing 不等于 computation 与 T 无关。


## 3. 经典两层 MLP 的数学

设 expansion width：

$$
M=4C.
$$

第一层扩大 feature space：

$$
h
=
xW_1+b_1,
$$

其中：

$$
W_1\in\mathbb R^{C\times4C},
\qquad
b_1\in\mathbb R^{4C}.
$$

经过非线性 activation：

$$
a=\phi(h).
$$

第二层压回 model width：

$$
\boxed{
y
=
aW_2+b_2
}
$$

其中：

$$
W_2\in\mathbb R^{4C\times C},
\qquad
b_2\in\mathbb R^C.
$$

对 batch/sequence tensor：

```text
X        (B,T,C)
↓ Linear C→4C
H        (B,T,4C)
↓ activation
A        (B,T,4C)
↓ Linear 4C→C
Y        (B,T,C)
```

`4C` 是经典、常见的 expansion ratio 教学选择，不是 Feed-Forward MLP 的数学硬性要求。


## 4. 为什么先 Expand 再 Contract？

如果一直只在 C 维空间里做一个简单线性映射：

$$
x\rightarrow xW
$$

能够表达的仍然只是 linear / affine transformation。

扩展到更宽的 hidden feature space：

$$
C\rightarrow4C
$$

给中间层更多 feature directions 来构造 nonlinear combinations；再压回：

$$
4C\rightarrow C
$$

使输出重新回到 Transformer 的 model width。

一个有用但不过度的直觉是：

> Attention 负责让 token 获得“来自哪里的信息”，MLP 再对当前 token 已经拥有的 features 做更丰富的 nonlinear processing。

这只是职责直觉，不意味着 Attention 完全不做 feature transformation，也不意味着 MLP 可以替代 Attention 的跨位置通信。


## 5. 为什么中间必须有 Activation？

假设暂时忽略 bias，而且没有 activation：

$$
xW_1W_2.
$$

定义：

$$
W_{eff}=W_1W_2.
$$

那么：

$$
xW_1W_2
=
xW_{eff}.
$$

两个 Linear 仍然等价于一个 Linear。

所以真正让两层 MLP 获得 nonlinear expressivity 的关键是：

$$
\boxed{
y
=
\phi(xW_1+b_1)W_2+b_2
}.
$$

经典 GPT-style MLP 常使用 GELU。MLX 0.32.2 提供：

```python
nn.GELU()
```

其标准定义：

$$
\operatorname{GELU}(x)
=
x\Phi(x),
$$

其中 `Phi` 是标准高斯分布的 CDF。

本节只使用 GELU 建立 MLP；不扩展到其他 gated MLP variants。


## 6. GELU 的最小直觉

GELU 和 hard threshold 不同，它是 smooth activation。

一些容易记的值：

$$
\operatorname{GELU}(0)=0,
$$

$$
\operatorname{GELU}(1)
\approx
0.8413,
$$

$$
\operatorname{GELU}(-1)
\approx
-0.1587,
$$

$$
\operatorname{GELU}(2)
\approx
1.9545.
$$

它不会像 ReLU 那样简单把所有负数直接设为 0；负值也可能以较小幅度通过。

本节不深入 GELU 的概率解释或近似算法，只需要知道它提供了 smooth nonlinearity。


## 7. Tiny Hand Example：`C=2 → 4 → 2`

取一个 token：

$$
x=[1,2].
$$

人为设计第一层：

$$
W_1
=
\begin{bmatrix}
1&-1&0&0\\
0&0&0&1
\end{bmatrix}.
$$

忽略 bias，则：

$$
h=xW_1=[1,-1,0,2].
$$

经过 GELU：

$$
a
\approx
[0.8413,-0.1587,0,1.9545].
$$

再取：

$$
W_2
=
\begin{bmatrix}
1&0\\
0&1\\
1&1\\
0.5&-0.5
\end{bmatrix}.
$$

则：

$$
y=aW_2.
$$

第一维约：

$$
0.8413+0.5(1.9545)
\approx
1.8186.
$$

第二维约：

$$
-0.1587-0.5(1.9545)
\approx
-1.1359.
$$

所以：

$$
\boxed{
y
\approx
[1.8186,-1.1359]
}.
$$

这个例子只是帮助追踪 `C→4C→activation→C`；真实参数由训练学习。


In [ ]:
import mlx.core as mx
import mlx.nn as nn

x = mx.array([[1.0, 2.0]])  # (1,C), C=2

W1 = mx.array([
    [1.0, -1.0, 0.0, 0.0],
    [0.0,  0.0, 0.0, 1.0],
])  # mathematical row-vector convention: (C,4C)

W2 = mx.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [0.5,-0.5],
])  # (4C,C)

gelu = nn.GELU()

h = x @ W1
a = gelu(h)
y = a @ W2

print('x:', x)
print('hidden:', h)
print('after GELU:', a)
print('y:', y)

# Expected approximately:
# hidden     = [1, -1, 0, 2]
# after GELU = [0.8413, -0.1587, 0, 1.9545]
# y          = [1.8186, -1.1359]
# This master notebook remains unexecuted.


## 8. MLX 0.32.2：一个最小 FeedForwardMLP

项目固定 `mlx==0.32.2`。

MLX `nn.Linear(input_dims, output_dims, bias=True)` 会对 input 的最后一个 feature axis 做 affine transformation，因此可以直接处理：

$$
(B,T,C).
$$

一个最小模块：

```python
class FeedForwardMLP(nn.Module):
    def __init__(self, dims):
        self.fc1 = nn.Linear(dims, 4 * dims)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims)

    def __call__(self, x):
        return self.fc2(self.act(self.fc1(x)))
```

这里没有 Python loop 遍历 T；同一个 Linear 自动应用于所有 batch/token positions 的最后一个 feature axis。


In [ ]:
class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(
            input_dims=dims,
            output_dims=4 * dims,
            bias=True,
        )
        self.act = nn.GELU()
        self.fc2 = nn.Linear(
            input_dims=4 * dims,
            output_dims=dims,
            bias=True,
        )

    def __call__(self, x: mx.array) -> mx.array:
        hidden = self.fc1(x)       # (B,T,C) -> (B,T,4C)
        hidden = self.act(hidden)  # (B,T,4C)
        out = self.fc2(hidden)     # (B,T,4C) -> (B,T,C)
        return out


B, T, C = 2, 4, 6
mlp = FeedForwardMLP(dims=C)
x_seq = mx.zeros((B, T, C))
mlp_out = mlp(x_seq)

print('input:', x_seq.shape)
print('fc1 stored weight:', mlp.fc1.weight.shape)
print('fc2 stored weight:', mlp.fc2.weight.shape)
print('output:', mlp_out.shape)

# Expected shapes:
# fc1.weight = (4C,C) = (24,6)
# fc2.weight = (C,4C) = (6,24)
# output     = (B,T,C) = (2,4,6)
# This master notebook remains unexecuted.


## 9. MLP 的 Parameters 有多少？

以代码中的 `bias=True` 为例。

第一层：

$$
C\rightarrow4C
$$

weight 数量：

$$
4C^2.
$$

bias：

$$
4C.
$$

第二层：

$$
4C\rightarrow C
$$

weight 数量：

$$
4C^2.
$$

bias：

$$
C.
$$

因此总计：

$$
\boxed{
8C^2+5C
}.
$$

如果两层都 `bias=False`，则只剩：

$$
\boxed{8C^2}
$$

个 weights。

这也说明 Transformer block 中，Feed-Forward MLP 往往占有相当可观的参数量。


## 10. 怎样承接 14.1–14.3？

Step 14.3 的 Pre-Norm generic skeleton 是：

$$
x_{new}
=
x+F(N(x)).
$$

现在我们终于有了第二种具体的 F：

```text
F = Feed-Forward MLP
```

所以从概念上可以写：

$$
x_{new}
=
x
+
\operatorname{MLP}(N(x)).
$$

这说明 MLP branch 也能使用：

- normalization；
- residual connection。

但是本节只定义 MLP sublayer 本身以及它如何作为 generic F；不会把 Attention branch + MLP branch + 两套 Norm + 两个 Residual 真正组装成完整 Block。

完整 assembly 属于 Step 14.5。


## 11. Step 14.4 材料边界

- Transformer Feed-Forward MLP 是 position-wise function：同一套 parameters 独立应用到每个 `(b,t)` 的 C-dimensional feature vector。
- 经典教学结构使用 `C -> 4C -> activation -> C`；4 倍 expansion 是常见设计选择，不是数学硬约束。
- 两层 Linear 中间必须有 nonlinear activation 才不会整体退化成单个 affine/linear map；本节使用 MLX 0.32.2 `nn.GELU()`。
- Attention 的核心职责是 cross-position mixing；MLP 的核心职责是 within-position nonlinear feature transformation。MLP 自身不建立 T×T token interaction。
- MLX `nn.Linear` 直接作用于最后一个 feature axis，因此 `(B,T,C)` 可以整体送入 MLP，无需 Python token loop。
- 当前 `bias=True` 的 `C->4C->C` 两层 MLP 参数量为 `8C^2+5C`；若 bias=False，则为 `8C^2`。
- MLP 输出恢复 `(B,T,C)`，因此后续可以参与 residual connection；本节只把它视为 14.3 generic F 的一个具体实例。
- 本轮只准备 Step 14.4；不进入 Step 14.5，不组装完整 Transformer Block，不计算整个 Block 的总参数，也不做 Block numerical sanity tests。
